## ECG v2

### Main code

In [147]:
def rerange_hex_file(input_path, output_path):
    """
    讀取 txt 檔，將內容補齊 4 位元，並兩兩合併成 8 位元 (小端順序) 後輸出新檔案

    input_path:  輸入檔案路徑
    output_path: 輸出檔案路徑
    """
    # 讀取檔案
    with open(input_path, "r") as f:
        lines = [line.strip() for line in f if line.strip()]

    # 兩兩合併 (小端順序)
    formatted_lines = []
    for i in range(0, len(lines), 2):
        first = lines[i].zfill(4)                             # 第一筆 (低位)
        second = lines[i+1].zfill(4) if i+1 < len(lines) else "0000"  # 第二筆 (高位，若不存在補 0000)
        combined = second + first                             # 小端：高位在前，低位在後
        formatted_lines.append(combined)

    # 寫入檔案
    with open(output_path, "w") as f:
        for line in formatted_lines:
            f.write(line + "\n")

    print(f"轉換完成 ✅ 輸出檔案: {output_path}")

In [148]:
def convert_file(input_path, output_path):
    output = []
    with open(input_path, "r") as f:
        lines = [line.strip() for line in f if line.strip()]

    # 每兩行合併成一行，低位在前，高位在後
    for i in range(0, len(lines), 2):
        low  = lines[i]      # 偶數行 (低位)
        high = lines[i+1]    # 奇數行 (高位)
        combined = f"{int(high, 16):04x}{int(low, 16):04x}"
        output.append(combined)

    with open(output_path, "w") as f:
        for line in output:
            f.write(line + "\n")

    print(f"轉換完成 ✅ 輸出檔案位置: {output_path}")

In [149]:
def rearrange_scan(input_path, output_path, scan_size):
    # 讀取檔案
    with open(input_path, "r") as f:
        data = [line.strip() for line in f if line.strip()]

    total = len(data)
    reordered = []

    # 掃描模式
    for offset in range(scan_size):
        i = offset
        while i < total:
            reordered.append(data[i])
            i += scan_size

    # 若資料筆數為奇數，自動補 0000
    if len(reordered) % 2 != 0:
        reordered.append("0000")

    # 每兩筆合併成32bit (高位在前)
    combined = []
    for i in range(0, len(reordered), 2):
        high = reordered[i + 1]
        low = reordered[i]
        combined.append(f"{high}{low}")

    # 輸出
    with open(output_path, "w") as f:
        for line in combined:
            f.write(line + "\n")

    print(f"轉換完成 ✅ 輸出檔案位置: {output_path}")


In [150]:
## all weight rerange
def rerange_weight_file(input_path, output_path):
    """
    讀取 weight 檔案 (每行可能含多組 16 進位數字)，
    將其拆成每組 4 位，並以小端順序兩兩合併成 8 位輸出。
    會自動忽略 // 註解行與空白行。
    """

    # 讀取檔案，忽略註解與空白行
    with open(input_path, "r") as f:
        lines = []
        for line in f:
            line = line.strip()
            if not line or line.startswith("//"):  # 跳過空白或註解
                continue
            # 若一行有內嵌註解，也去除掉
            if "//" in line:
                line = line.split("//")[0].strip()
            lines.append(line)

    # 拆成 4 位一組
    groups = []
    for line in lines:
        for i in range(0, len(line), 4):
            group = line[i:i+4].zfill(4)  # 不足 4 位補 0
            groups.append(group)

    # 每兩組合併，小端順序
    formatted_lines = []
    for i in range(0, len(groups), 2):
        first = groups[i]
        second = groups[i+1] if i+1 < len(groups) else "0000"
        combined = second + first
        formatted_lines.append(combined)

    # 輸出結果
    with open(output_path, "w") as f:
        for line in formatted_lines:
            f.write(line + "\n")

    print(f"✅ 轉換完成！輸出檔案：{output_path}")

In [151]:
def convert_hex_by_scan(input_path, output_path, scan_lines=4):
    # 讀取輸入
    with open(input_path, 'r') as f:
        lines = [line.strip() for line in f if line.strip()]

    # 每行拆成 4 字元一組
    split_lines = [[line[i:i+4] for i in range(0, len(line), 4)] for line in lines]

    result = []

    # 每次處理 scan_lines 行
    for i in range(0, len(split_lines), scan_lines):
        group = split_lines[i:i+scan_lines]
        if len(group) < scan_lines:
            break  # 不足一組就跳出

        num_cols = len(group[0])
        # 對每一欄操作
        for col in range(num_cols):
            # 每兩行一組（下 + 上）
            for row in range(0, scan_lines, 2):
                high = group[row+1][col]
                low = group[row][col]
                result.append(high + low)

    # 輸出
    with open(output_path, 'w') as f:
        for r in result:
            f.write(r + '\n')

    print(f"✅ 轉換完成（掃描數={scan_lines}）→ {output_path}")



In [152]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

def convert_hex_nonoverlap_pairs_all_bases(input_path, output_path, unit_chars=4, jump_lines=2):
    """
    unit_chars: 每組字元數（預設 4）。
    jump_lines: 每次跳行數（例如 2）。
    對 base = 0..jump_lines-1 每個起始列，針對每個 group index 建序列:
      S = [base, base + jump, base + 2*jump, ...]
    對序列取不重疊鄰對: (S[1],S[0]), (S[3],S[2]), ...
    輸出 high+low（每行一筆）。
    """
    # 讀檔
    with open(input_path, 'r', encoding='utf-8') as f:
        raw_lines = [ln.rstrip('\n').strip() for ln in f if ln.strip()]

    if not raw_lines:
        print("輸入檔為空。")
        return

    # 每行切成 unit_chars 一組
    split_lines = [[ln[i:i+unit_chars] for i in range(0, len(ln), unit_chars)] for ln in raw_lines]
    num_lines = len(split_lines)
    num_groups = max(len(r) for r in split_lines)

    # 若某行不足 group 數，用空字串補齊（避免 index error）
    for r in split_lines:
        if len(r) < num_groups:
            r.extend([''] * (num_groups - len(r)))

    results = []

    # 針對每個可能的 base（0 .. jump_lines-1）
    for base in range(jump_lines):
        # 對每個 group index 處理
        for g in range(num_groups):
            # 建序列 S: base, base+jump, base+2*jump, ...
            seq = []
            cur = base
            while cur < num_lines:
                val = split_lines[cur][g]
                # 若該位置為空字串（表示該行在該 group 沒資料），跳過但仍繼續序列
                seq.append(val)
                cur += jump_lines

            # 取不重疊相鄰 pair：(seq[1],seq[0]), (seq[3],seq[2]), ...
            i = 1
            while i < len(seq):
                high = seq[i]
                low  = seq[i-1]
                if high and low:
                    results.append(high + low)
                i += 2

    # 寫出結果
    with open(output_path, 'w', encoding='utf-8') as f:
        for r in results:
            f.write(r + '\n')

    print(f"✅ 轉換完成（跳行={jump_lines}）→ {output_path}")

In [156]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

def reorder_by_jump_lines_file_simple(input_path, output_path, jump_lines=10):
    """
    每行一筆資料，不拆分，每 jump_lines 行重新排列。
    """
    with open(input_path, 'r', encoding='utf-8') as f:
        data = [line.strip() for line in f if line.strip()]

    num_lines = len(data)
    result = []

    # 對每個 base 行
    for base in range(jump_lines):
        idx = base
        while idx < num_lines:
            result.append(data[idx])
            idx += jump_lines

    with open(output_path, 'w', encoding='utf-8') as f:
        for line in result:
            f.write(line + '\n')
    print(f"✅ 轉換完成（跳行={jump_lines}）→ {output_path}")

### ECG v2 conv out

In [123]:
## conv1 output
convert_file("./ecg_v2/golden/conv1_out.txt", "./ecg_v2_rerange/golden/conv1_out_rerange.txt")

轉換完成 ✅ 輸出檔案位置: ./ecg_v2_rerange/golden/conv1_out_rerange.txt


In [ ]:
## conv2 output
rearrange_scan("./ecg_v2/golden/conv2_out.txt", "./ecg_v2_rerange/golden/conv2_out_rerange.txt", scan_size=64)
## conv2 output part
rearrange_scan("./ecg_v2/golden/conv2_p0_out.txt", "./ecg_v2_rerange/golden/conv2_p0_out_rerange.txt", scan_size=64)
rearrange_scan("./ecg_v2/golden/conv2_p1_out.txt", "./ecg_v2_rerange/golden/conv2_p1_out_rerange.txt", scan_size=64)
rearrange_scan("./ecg_v2/golden/conv2_p2_out.txt", "./ecg_v2_rerange/golden/conv2_p2_out_rerange.txt", scan_size=64)
rearrange_scan("./ecg_v2/golden/conv2_p3_out.txt", "./ecg_v2_rerange/golden/conv2_p3_out_rerange.txt", scan_size=64)

轉換完成 ✅ 輸出檔案位置: ./ecg_v2_rerange/golden/conv2_out_rerange.txt
轉換完成 ✅ 輸出檔案位置: ./ecg_v2_rerange/golden/conv2_p0_out_rerange.txt
轉換完成 ✅ 輸出檔案位置: ./ecg_v2_rerange/golden/conv2_p1_out_rerange.txt
轉換完成 ✅ 輸出檔案位置: ./ecg_v2_rerange/golden/conv2_p2_out_rerange.txt
轉換完成 ✅ 輸出檔案位置: ./ecg_v2_rerange/golden/conv2_p3_out_rerange.txt


In [155]:
## conv3 output
rearrange_scan("./ecg_v2/golden/conv3_out.txt", "./ecg_v2_rerange/golden/conv3_out_rerange.txt", scan_size=64)
## conv3 output part
rearrange_scan("./ecg_v2/golden/conv3_p0_out.txt", "./ecg_v2_rerange/golden/conv3_p0_out_rerange.txt", scan_size=64)
rearrange_scan("./ecg_v2/golden/conv3_p1_out.txt", "./ecg_v2_rerange/golden/conv3_p1_out_rerange.txt", scan_size=64)


轉換完成 ✅ 輸出檔案位置: ./ecg_v2_rerange/golden/conv3_out_rerange.txt
轉換完成 ✅ 輸出檔案位置: ./ecg_v2_rerange/golden/conv3_p0_out_rerange.txt
轉換完成 ✅ 輸出檔案位置: ./ecg_v2_rerange/golden/conv3_p1_out_rerange.txt


In [125]:
# fc1 input
rearrange_scan("./ecg_v2/golden/fc1_in.txt", "./ecg_v2_rerange/golden/fc1_in_rerange.txt", scan_size=1)

轉換完成 ✅ 輸出檔案位置: ./ecg_v2_rerange/golden/fc1_in_rerange.txt


In [126]:
# fc1 output
rearrange_scan("./ecg_v2/golden/dense1_out.txt", "./ecg_v2_rerange/golden/fc1_out_rerange.txt", scan_size=10)
# fc2 output
rearrange_scan("./ecg_v2/golden/dense2_out.txt", "./ecg_v2_rerange/golden/fc2_out_rerange.txt", scan_size=5)

轉換完成 ✅ 輸出檔案位置: ./ecg_v2_rerange/golden/fc1_out_rerange.txt
轉換完成 ✅ 輸出檔案位置: ./ecg_v2_rerange/golden/fc2_out_rerange.txt


### ECG v2 input

In [127]:
## conv1 input
rerange_hex_file("./ecg_v2/input/input1.txt", "./ecg_v2_rerange/input/conv1_input_rerange.txt")

轉換完成 ✅ 輸出檔案: ./ecg_v2_rerange/input/conv1_input_rerange.txt


### ECG v2 weight

In [128]:
## all weight
rerange_weight_file("./ecg_v2/weight/weight.txt", "./ecg_v2_rerange/weight/weight_rerange.txt")

✅ 轉換完成！輸出檔案：./ecg_v2_rerange/weight/weight_rerange.txt


In [ ]:
## conv2 weight v3
convert_hex_by_scan("./ecg_v2/weight/conv2_p0_weight.txt", "./ecg_v2_rerange/weight/conv2_p0_weight_rerange.txt", scan_lines=64)
convert_hex_by_scan("./ecg_v2/weight/conv2_p1_weight.txt", "./ecg_v2_rerange/weight/conv2_p1_weight_rerange.txt", scan_lines=64)
convert_hex_by_scan("./ecg_v2/weight/conv2_p2_weight.txt", "./ecg_v2_rerange/weight/conv2_p2_weight_rerange.txt", scan_lines=64)
convert_hex_by_scan("./ecg_v2/weight/conv2_p3_weight.txt", "./ecg_v2_rerange/weight/conv2_p3_weight_rerange.txt", scan_lines=64)

✅ 轉換完成（掃描數=64）→ ./ecg_v2_rerange/weight/conv2_p0_weight_rerange.txt
✅ 轉換完成（掃描數=64）→ ./ecg_v2_rerange/weight/conv2_p1_weight_rerange.txt
✅ 轉換完成（掃描數=64）→ ./ecg_v2_rerange/weight/conv2_p2_weight_rerange.txt
✅ 轉換完成（掃描數=64）→ ./ecg_v2_rerange/weight/conv2_p3_weight_rerange.txt


In [158]:
## conv3 weight
convert_hex_by_scan("./ecg_v2/weight/conv3_p0_weight.txt", "./ecg_v2_rerange/weight/conv3_p0_weight_rerange.txt", scan_lines=64)
convert_hex_by_scan("./ecg_v2/weight/conv3_p1_weight.txt", "./ecg_v2_rerange/weight/conv3_p1_weight_rerange.txt", scan_lines=64)

✅ 轉換完成（掃描數=64）→ ./ecg_v2_rerange/weight/conv3_p0_weight_rerange.txt
✅ 轉換完成（掃描數=64）→ ./ecg_v2_rerange/weight/conv3_p1_weight_rerange.txt


In [146]:
# fc1
# convert_hex_nonoverlap_pairs_all_bases('./ecg_v2/weight/dense1_weight.txt', './ecg_v2_rerange/weight/dense1_weight_rerange.txt', jump_lines=4)
reorder_by_jump_lines_file_simple('./ecg_v2_rerange/weight/dense1_weight_rerange.txt', './ecg_v2_rerange/weight/dense1_weight_rerange_2.txt', jump_lines=36)

✅ 轉換完成（跳行=36）→ ./ecg_v2_rerange/weight/dense1_weight_rerange_2.txt


In [139]:
# fc2
convert_hex_nonoverlap_pairs_all_bases('./ecg_v2/weight/dense2_weight.txt', './ecg_v2_rerange/weight/dense2_weight_rerange.txt', jump_lines=2)

✅ 轉換完成（跳行=2）→ ./ecg_v2_rerange/weight/dense2_weight_rerange.txt
